In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadshahidazeem/customer-churn-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\abdil\.cache\kagglehub\datasets\muhammadshahidazeem\customer-churn-dataset\versions\1


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
import xgboost as xgb

In [4]:
df_train = pd.read_csv(path + "/customer_churn_dataset-training-master.csv")
df_test = pd.read_csv(path + "/customer_churn_dataset-testing-master.csv")

In [5]:
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

In [6]:
df_train.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,71073.0,27.0,Male,14.0,28.0,3.0,16.0,Standard,Monthly,862.00,9.0,1.0
1,230192.0,40.0,Male,19.0,2.0,8.0,28.0,Standard,Monthly,620.81,21.0,1.0
2,22407.0,27.0,Female,57.0,3.0,0.0,24.0,Standard,Annual,915.00,26.0,1.0
3,290822.0,40.0,Male,21.0,14.0,0.0,11.0,Basic,Annual,592.83,9.0,0.0
4,172430.0,39.0,Male,58.0,4.0,2.0,8.0,Standard,Monthly,694.00,15.0,1.0


In [7]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


In [8]:
df_train.describe()

,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567107
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [9]:
df_train = df_train.drop("CustomerID", axis=1)
df_test = df_test.drop("CustomerID", axis=1)

In [10]:
print(df_train["Contract Length"].isna().sum())
print(df_test["Contract Length"].isna().sum())

1
0


In [11]:
df_train = df_train.dropna(subset=['Contract Length'])
df_test = df_test.dropna(subset=['Contract Length'])

In [12]:
df_train['Contract Length'].unique

<bound method Series.unique of 0         Monthly
1         Monthly
2          Annual
3          Annual
4         Monthly
           ...   
440828     Annual
440829     Annual
440830    Monthly
440831     Annual
440832     Annual
Name: Contract Length, Length: 440832, dtype: object>

In [13]:
df_train['Contract Length'] = df_train['Contract Length'].map({"Annual": 12, "Monthly": 1, "Quarterly": 3})
df_test['Contract Length'] = df_test['Contract Length'].map({"Annual": 12, "Monthly": 1, "Quarterly": 3})

In [14]:
df_train['Contract Length'].unique()

array([ 1, 12,  3])

In [15]:
df_train['Gender'].unique()

array(['Male', 'Female'], dtype=object)

In [16]:
df_train['Gender'] = df_train['Gender'].map({"Female": 1, "Male": 0})
df_test['Gender'] = df_test['Gender'].map({"Female": 1, "Male": 0})

In [17]:
df_train['Gender'].unique()

array([0, 1])

In [18]:
df_train = pd.get_dummies(df_train, prefix=["Subscription Type"], dtype=float)
df_test = pd.get_dummies(df_test, prefix=["Subscription Type"], dtype=float)

In [19]:
X_train = df_train.drop('Churn', axis=1)
y_train = df_train['Churn']

X_test = df_test.drop('Churn', axis=1)
y_test = df_test['Churn'].astype(float)

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.22      0.36     33881
         1.0       0.53      0.99      0.69     30493

    accuracy                           0.58     64374
   macro avg       0.74      0.60      0.53     64374
weighted avg       0.75      0.58      0.52     64374



In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_jobs=-1)

rf.fit(X_train_scaled, y_train)

y_pred = rf.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.98      0.06      0.11     33881
         1.0       0.49      1.00      0.66     30493

    accuracy                           0.50     64374
   macro avg       0.73      0.53      0.38     64374
weighted avg       0.75      0.50      0.37     64374



In [23]:
print(rf.predict(X_test_scaled[:20]))
print(y_test[:20].values)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [24]:
print(X_train.head(10))

    Age  Gender  Tenure  Usage Frequency  Support Calls  Payment Delay  \
0  27.0       0    14.0             28.0            3.0           16.0   
1  40.0       0    19.0              2.0            8.0           28.0   
2  27.0       1    57.0              3.0            0.0           24.0   
3  40.0       0    21.0             14.0            0.0           11.0   
4  39.0       0    58.0              4.0            2.0            8.0   
5  24.0       0    49.0             29.0            2.0            5.0   
6  35.0       0    25.0             21.0            2.0            1.0   
7  52.0       1    50.0             22.0           10.0            4.0   
8  41.0       1    20.0             22.0            0.0           15.0   
9  22.0       1    33.0             24.0            0.0           11.0   

   Contract Length  Total Spend  Last Interaction  Subscription Type_Basic  \
0                1       862.00               9.0                      0.0   
1                1       620.

In [25]:
print(y_train[:50].values)

[1. 1. 1. 0. 1. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 1.
 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 0.
 1. 1.]


In [26]:
print(X_train.shape)
print(X_test.shape)
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(440832, 12)
(64374, 12)
(440832, 12)
(64374, 12)


In [27]:
print(rf.predict_proba(X_test_scaled[:10]))

[[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]]


In [28]:
rf2 = RandomForestClassifier(n_jobs=-1, random_state=42)
rf2.fit(X_train_scaled, y_train)
print(rf2.predict_proba(X_test_scaled[:10]))

[[0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]
 [0. 1.]]


In [29]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Quick sanity check - can RF learn anything on a small sample?
X_sample = X_train_scaled[:1000]
y_sample = y_train.values[:1000]

print("y_sample unique:", np.unique(y_sample))

rf_test = RandomForestClassifier(n_jobs=-1, random_state=42)
rf_test.fit(X_sample, y_sample)
print(rf_test.predict_proba(X_test_scaled[:5]))

y_sample unique: [0. 1.]
[[0.   1.  ]
 [0.01 0.99]
 [0.1  0.9 ]
 [0.   1.  ]
 [0.01 0.99]]


In [30]:
print(y_train.values[:20])
print(y_train.values[-20:])

[1. 1. 1. 0. 1. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 0.]
[1. 0. 1. 0. 1. 1. 0. 0. 1. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1.]


In [31]:
# 1. Shuffle
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Split AFTER shuffle
X_train = df_train.drop('Churn', axis=1)
y_train = df_train['Churn']

# 3. Scale AFTER split
X_train_scaled = scaler.fit_transform(X_train)

# 4. Then fit
rf2 = RandomForestClassifier(n_jobs=-1, random_state=42)
rf2.fit(X_train_scaled, y_train)
print(rf2.predict_proba(X_test_scaled[:10]))

[[0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.01 0.99]]


In [32]:
print(y_train.values[:20])
print(y_train.values[100000:100020])
print(y_train.values[200000:200020])

[1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1. 1. 0. 1.]
[1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 0. 1. 1. 0. 1. 0. 0. 1. 0.]


In [33]:
print(y_train.value_counts())

Churn
1.0    249999
0.0    190833
Name: count, dtype: int64


In [34]:
rf3 = RandomForestClassifier(n_jobs=-1, random_state=42)
rf3.fit(X_train, y_train)  # raw, unscaled
print(rf3.predict_proba(X_test[:10]))

[[0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.   1.  ]
 [0.01 0.99]]


In [35]:
for size in [1000, 5000, 10000, 50000, 100000]:
    rf_t = RandomForestClassifier(n_jobs=-1, random_state=42)
    rf_t.fit(X_train.iloc[:size], y_train.iloc[:size])
    preds = rf_t.predict(X_test)
    from sklearn.metrics import accuracy_score
    print(f"Size {size}: accuracy={accuracy_score(y_test, preds):.3f}, pred_1_ratio={preds.mean():.3f}")

Size 1000: accuracy=0.520, pred_1_ratio=0.952
Size 5000: accuracy=0.511, pred_1_ratio=0.961
Size 10000: accuracy=0.509, pred_1_ratio=0.963
Size 50000: accuracy=0.506, pred_1_ratio=0.966
Size 100000: accuracy=0.504, pred_1_ratio=0.968


In [36]:
print(X_test.head())
print(X_test.dtypes)

   Age  Gender  Tenure  Usage Frequency  Support Calls  Payment Delay  \
0   22       1      25               14              4             27   
1   41       1      28               28              7             13   
2   47       0      27               10              2             29   
3   35       0       9               12              5             17   
4   53       1      58               24              9              2   

   Contract Length  Total Spend  Last Interaction  Subscription Type_Basic  \
0                1          598                 9                      1.0   
1                1          584                20                      0.0   
2               12          757                21                      0.0   
3                3          232                18                      0.0   
4               12          533                18                      0.0   

   Subscription Type_Premium  Subscription Type_Standard  
0                        0.0     

In [37]:
print(X_train['Total Spend'].describe())
print(X_test['Total Spend'].describe())

count    440832.000000
mean        631.616223
std         240.803001
min         100.000000
25%         480.000000
50%         661.000000
75%         830.000000
max        1000.000000
Name: Total Spend, dtype: float64
count    64374.000000
mean       541.023379
std        260.874809
min        100.000000
25%        313.000000
50%        534.000000
75%        768.000000
max       1000.000000
Name: Total Spend, dtype: float64


In [38]:
print(X_train['Support Calls'].describe())
print(X_test['Support Calls'].describe())

count    440832.000000
mean          3.604437
std           3.070218
min           0.000000
25%           1.000000
50%           3.000000
75%           6.000000
max          10.000000
Name: Support Calls, dtype: float64
count    64374.000000
mean         5.400690
std          3.114005
min          0.000000
25%          3.000000
50%          6.000000
75%          8.000000
max         10.000000
Name: Support Calls, dtype: float64


In [39]:
df_combined = pd.concat([df_train, df_test], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [41]:
X = df_combined.drop('Churn', axis=1)
y = df_combined['Churn']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [45]:
rf = RandomForestClassifier(n_jobs=-1)

rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       1.00      0.86      0.92     44952
         1.0       0.90      1.00      0.95     56090

    accuracy                           0.94    101042
   macro avg       0.95      0.93      0.93    101042
weighted avg       0.94      0.94      0.94    101042



In [46]:
y_train_pred = rf.predict(X_train_scaled)
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    179762
         1.0       1.00      1.00      1.00    224402

    accuracy                           1.00    404164
   macro avg       1.00      1.00      1.00    404164
weighted avg       1.00      1.00      1.00    404164



In [50]:
rf2 = RandomForestClassifier(n_jobs=-1, random_state=42, max_depth=5)
rf2.fit(X_train_scaled, y_train)
y_pred = rf2.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.86      0.90     44952
         1.0       0.89      0.96      0.93     56090

    accuracy                           0.92    101042
   macro avg       0.92      0.91      0.91    101042
weighted avg       0.92      0.92      0.92    101042



In [51]:
y_train_pred = rf.predict(X_train_scaled)
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    179762
         1.0       1.00      1.00      1.00    224402

    accuracy                           1.00    404164
   macro avg       1.00      1.00      1.00    404164
weighted avg       1.00      1.00      1.00    404164



In [52]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_jobs=-1, random_state=42)
xgb.fit(X_train_scaled, y_train)
y_pred = xgb.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.86      0.92     44952
         1.0       0.90      0.99      0.94     56090

    accuracy                           0.93    101042
   macro avg       0.94      0.93      0.93    101042
weighted avg       0.94      0.93      0.93    101042



In [55]:
y_train_pred = xgb.predict(X_train_scaled)
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.86      0.92    179762
         1.0       0.90      1.00      0.94    224402

    accuracy                           0.93    404164
   macro avg       0.95      0.93      0.93    404164
weighted avg       0.94      0.93      0.93    404164

